# 02 Alignment (B-11)

Summaries with confidence intervals, paired tests, and the report figure. Reads `results/scores.csv`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from humper import data, evaluate

OUT = data.REPO_ROOT / "results"
FIG = OUT / "figures" / "alignment"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE = "#2a6fb0", "#c46a1f"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

scores = pd.read_csv(OUT / "scores.csv")
table = evaluate.summary(scores)
table.to_csv(TAB / "alignment_summary.csv", index=False)
table.pivot(index="explainer", columns="metric", values="mean").round(3)

In [ ]:
EXPLAINERS = ["resnet50_gradcam", "vit_gradcam", "vit_rollout"]
METRICS = ["cc", "nss", "sauc"]
pairs = [(e, "center") for e in EXPLAINERS]
pairs += [(e, f"shuffled_{e}") for e in EXPLAINERS]
pairs += [("resnet50_gradcam", "vit_gradcam"), ("vit_gradcam", "vit_rollout"), ("resnet50_gradcam", "vit_gradcam_7x7")]
tests = pd.DataFrame([evaluate.paired(scores, a, b, m) for a, b in pairs for m in METRICS])
tests["holm_p"] = evaluate.holm(tests.wilcoxon_p)
tests.to_csv(TAB / "paired_tests.csv", index=False)
tests.round(4)

In [ ]:
ROWS = [("resnet50_gradcam", "ResNet-50 Grad-CAM"), ("vit_gradcam", "ViT Grad-CAM"), ("vit_rollout", "ViT Rollout"),
        ("center", "Center prior"), ("shuffled_resnet50_gradcam", "Shuffled ResNet-50"),
        ("shuffled_vit_gradcam", "Shuffled ViT Grad-CAM"), ("shuffled_vit_rollout", "Shuffled ViT Rollout")]
TITLES = {"cc": "CC", "nss": "NSS", "sauc": "Shuffled AUC"}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), sharey=True)
positions = {name: len(ROWS) - 1 - i for i, (name, _) in enumerate(ROWS)}
for ax, metric in zip(axes, METRICS):
    part = table[table.metric == metric].set_index("explainer")
    for name, _ in ROWS:
        r = part.loc[name]
        explainer = name in EXPLAINERS
        ax.errorbar(r["mean"], positions[name], xerr=[[r["mean"] - r.ci_low], [r.ci_high - r["mean"]]],
                    fmt="o" if explainer else "s", color=BLUE if explainer else ORANGE, markersize=7, capsize=0, linewidth=2)
    ax.set_title(TITLES[metric]); ax.grid(axis="x", color="#e5e5e5", linewidth=0.8); ax.set_axisbelow(True)
    if metric == "sauc":
        ax.axvline(0.5, color="#777777", linewidth=1, linestyle=":")
    else:
        ax.axvline(0, color="#777777", linewidth=1, linestyle=":")
axes[0].set_yticks(list(positions.values()), [label for _, label in ROWS])
handles = [plt.Line2D([], [], marker="o", color=BLUE, linestyle="", markersize=7, label="Explanation"),
           plt.Line2D([], [], marker="s", color=ORANGE, linestyle="", markersize=7, label="Reference")]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(FIG / "alignment_means.png", dpi=200); plt.show()

In [ ]:
SHOW = [("resnet50_gradcam", "ResNet-50 Grad-CAM"), ("vit_gradcam", "ViT Grad-CAM"), ("vit_rollout", "ViT Rollout"), ("center", "Center prior")]
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharex=True, sharey=True)
for ax, (name, label) in zip(axes, SHOW):
    values = scores[scores.explainer == name].cc
    ax.hist(values, bins=np.arange(-0.8, 1.01, 0.1), color=BLUE if name != "center" else ORANGE, rwidth=0.88)
    ax.axvline(values.mean(), color="#222222", linewidth=1.5)
    ax.set_title(f"{label}\nmean {values.mean():.2f}", fontsize=10); ax.set_xlabel("CC per image")
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.8); ax.set_axisbelow(True)
axes[0].set_ylabel("Images")
fig.tight_layout(); fig.savefig(FIG / "cc_distribution.png", dpi=200); plt.show()

## Does the explanation resemble the generic center prior?

Tests the hypothesis that Rollout puts weight away from where people usually look.

In [ ]:
similarity = evaluate.center_similarity(data.load_images())
result = similarity.groupby("explainer").cc_with_center.agg(["mean", "median", lambda s: (s < 0).mean()]).rename(columns={"<lambda_0>": "share_negative"})
result.to_csv(TAB / "center_similarity.csv")
result.round(3)